In [1]:
# ! pip install -q -U bitsandbytes
! pip install --upgrade -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [2]:
import torch
from getpass import getpass

from diffusers import DiffusionPipeline, FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig

from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig
from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Нужно получить доступ к репозиторию с моделью:
https://huggingface.co/black-forest-labs/FLUX.1-dev

1. Зарегистрироваться на HF
2. В профиле создать токен
3. Перейти в репозиторий и запросить доступ (будет выдан автоматически)

In [3]:
HF_TOKEN = getpass()

··········


In [4]:
base_model = "black-forest-labs/FLUX.1-dev"

In [5]:
bnb_config = PipelineQuantizationConfig(
    quant_mapping={
        "transformer": DiffusersBitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        ),
        "text_encoder_2": TransformersBitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        ),
    }
)

In [6]:
pipe = FluxPipeline.from_pretrained(base_model,
                                         torch_dtype=torch.bfloat16,
                                         quantization_config=bnb_config,
                                         token=HF_TOKEN)


lora_repo = "strangerzonehf/Flux-Icon-Kit-LoRA"
trigger_word = "Icon Kit"
pipe.load_lora_weights(lora_repo)

device = torch.device("cuda")
pipe.to(device)

model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Icon-Kit.safetensors:   0%|          | 0.00/613M [00:00<?, ?B/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


FluxPipeline {
  "_class_name": "FluxPipeline",
  "_diffusers_version": "0.37.1",
  "_name_or_path": "black-forest-labs/FLUX.1-dev",
  "feature_extractor": [
    null,
    null
  ],
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "FlowMatchEulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "text_encoder_2": [
    "transformers",
    "T5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "T5Tokenizer"
  ],
  "transformer": [
    "diffusers",
    "FluxTransformer2DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [8]:
image = pipe(
    prompt="Icon Kit, minimalist app icon, magnifying glass, flat design, clean background",
    height=512,
    width=512,
    num_inference_steps=28,
    guidance_scale=3.5,
).images[0]

image.show()


  0%|          | 0/28 [00:00<?, ?it/s]

In [10]:
image.save("icon_output.png")

In [11]:
! pip freeze > requirement.txt